In [ ]:
!unzip -q "/content/digits.zip"
!unzip -q "/content/THDigits.zip"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F            adds some efficiency
from torch.utils.data import DataLoader, TensorDataset    lets us load data in batches
from torch.utils.data import Subset        it is used to split our data
from torchvision import datasets, transforms
from torchvision.transforms.functional import to_pil_image
from torchsummary import summary

from sklearn.model_selection import train_test_split   it is used to split our data

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset

class ArabicToThaiImageDataset(Dataset):
    def __init__(self, arabic_dir, thai_dir, transform=None):
        self.arabic_dir = arabic_dir
        self.thai_dir = thai_dir
        self.transform = transform

          Scan all digit folders (e.g. "0", "1", ...)
        digit_folders = [f for f in os.listdir(arabic_dir) if os.path.isdir(os.path.join(arabic_dir, f))]

        self.pairs = []

        for digit_folder in digit_folders:
            arabic_subdir = os.path.join(arabic_dir, digit_folder)
            thai_subdir = os.path.join(thai_dir, digit_folder)
            if not os.path.isdir(thai_subdir):
                continue    skip if no matching Thai folder

              List all image files in arabic and thai subfolders
            arabic_images = sorted([
                f for f in os.listdir(arabic_subdir)
                if os.path.isfile(os.path.join(arabic_subdir, f)) and f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ])

            thai_images = sorted([
                f for f in os.listdir(thai_subdir)
                if os.path.isfile(os.path.join(thai_subdir, f)) and f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ])

              Find matching files by name after removing "TH" prefix from Thai filenames
            matching_images = [f for f in arabic_images if f.replace('TH','') in thai_images or f in [thai_fname.replace('TH','') for thai_fname in thai_images]]


            for fname in matching_images:
                  Determine the correct Thai filename with the "TH" prefix
                thai_fname = f'TH{fname}' if f'TH{fname}' in thai_images else fname
                arabic_path = os.path.join(arabic_subdir, fname)
                thai_path = os.path.join(thai_subdir, thai_fname)
                self.pairs.append((arabic_path, thai_path))


        if len(self.pairs) == 0:
            raise ValueError("No matching image pairs found in the directories.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        arabic_path, thai_path = self.pairs[idx]

        arabic_img = Image.open(arabic_path).convert("RGB")
        thai_img = Image.open(thai_path).convert("RGB")

        if self.transform:
            arabic_img = self.transform(arabic_img)
            thai_img = self.transform(thai_img)

        return arabic_img, thai_img



from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(),
    transforms.ToTensor(),
    lambda x: 1 - x
])

dataset = ArabicToThaiImageDataset("/content/digits", "/content/THDigits", transform=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

for arabic_imgs, thai_imgs in dataloader:
    print(arabic_imgs.shape, thai_imgs.shape)
    break